### Code to extract documents for all chosen source domains in preparation for metaphor analysis

Chosen source domains:
- Cognitive Agency:
    - trainee -> Auszubildender
    - god -> Gott
- Behavioral Agency:
    - conversational partner: child, friend -> Kind, Freund
- Labor:
    - artist -> Künstler
    - assistant -> Assistent
    - doctor -> Arzt
    - (expert -> Experte)
    - teacher -> Lehrer
    - thief -> Dieb
    - programmer -> Programmierer
    - writer -> Author
    - (co)worker -> Kollege (Mitarbeiter)

In [ ]:
import pandas as pd
from pprint import pprint
import os
import re
import json
import matplotlib.pyplot as plt
import ast

import spacy
from spacy.matcher import DependencyMatcher
from spacy import displacy

nlp = spacy.load('de_core_news_lg') # oder de_core_news_sm

def flatten(xss):
    return [x for xs in xss for x in xs]

plots_dir = "plots/"
try:
    os.mkdir(plots_dir)
    print(f"Directory '{plots_dir}' created successfully.")
except FileExistsError:
    print(f"Directory '{plots_dir}' already exists.")

Directory 'plots/' already exists.


In [ ]:
# preprocessing auxiliary functions

stopwords = open("config_files/spacy_stopwords.txt").read().replace("{", '[').replace("}",']')
stopwords = ast.literal_eval(stopwords)

# this one is not needed if working on Sentence_spacy
def remove_punctuation(corpus):
    cleaned_corpus = corpus.translate(str.maketrans('', '', string.punctuation))
    return(cleaned_corpus)

# remove stop words from text
def remove_stop_words_from_text(cleaned_corpus):
    out = [word for word in cleaned_corpus if not word in stopwords]
    out_text = ' '.join(out)
    return out_text  # , out

# search with stemm/lemma
def get_lemma(sent):
    doc = nlp(sent)
    return [token.lemma_ for token in doc]

In [9]:
# list of domains
source_doms = ['Auszubildender', 'Gott', 'Kind', 'Freund', 'Künstler', 'Assistent', 'Arzt', 'Lehrer', 'Dieb', 'Programmierer', 'Author', 'Kollege']
source_doms_lemma = [get_lemma(x.lower())[0] for x in source_doms]
print(source_doms_lemma)

['auszubildender', 'Gott', 'Kind', 'Freund', 'Künstler', 'Assistent', 'Arzt', 'Lehrer', 'Dieb', 'Programmierer', 'Author', 'Kollege']


In [ ]:
# TODO: build regex(s) for search, use lexicon?, TFIDF?
regex_d = {}

#### RegEx search for sentences 

In [ ]:
# 1. build lexcons for each domain, label some examples per domain using regex search
# get sentences
def to_sentences(sentences_file):
    sentences = []
    with open(sentences_file) as f:
        for line in f:
            txt = remove_stop_words_from_text(remove_punctuation(line.lower()))
            txt = ''.join(get_lemma(txt))
            print(txt)
            sentences.append(txt)  
    return sentences

# get sents in Leipzig data
leipzig_main = "deu_news"
sentences = {}
sentences_all = pd.DataFrame(columns=['id', 'sentence'])
for p in os.listdir(leipzig_main):
    k = p.replace("deu_news_","")
    sentences_file = os.path.join(leipzig_main, p, f"{p}-sentences.txt")
    # extract sentences
    sentences[k] = to_sentences(sentences_file)
    for sent in sentences[k]:       
        # TODO: use correct regex_d
        if len(sent) > 0 and re.search(regex_d, sent):
            split = re.sub(r"\t", "-", sent).split("-")
            sentences_all.loc[len(sentences_all)]= [f'{k}_{split[0]}',split[1]]

#TODO: reduce examples to top_n based on similarity


#### Cluster sentences for each domain

In [ ]:
# use to cluster, save files
source_d_path = "source_domains"
dom_sentences = pd.DataFrame(columns=['id','domain','sentence'])
for p in source_doms_lemma:
    # TODO: cluster domains, return sentence(s)

    # to df
    s_id = 0
    sentence = ''
    dom_sentences.loc[len(dom_sentences)] = [s_id, p, sentence]
    # save to file
    dom_sentences.to_csv(os.path.join(source_d_path, p, f"{p}_sentences.txt"))